# Tesla Deliveries - End-to-End ML Pipeline (2015-2025)

| # | Section | Description |
|---|---------|-------------|
| 1 | Data Loading & Inspection | Load CSV, schema validation, summary statistics |
| 2 | Data Preprocessing | Missing values, duplicates, outliers, encoding |
| 3 | Exploratory Data Analysis | Distributions, correlations, trends, seasonality |
| 4 | Feature Engineering | Lag features, rolling stats, cyclical encoding, interactions |
| 5 | Regression Modeling | Linear, Ridge, Lasso, Random Forest, XGBoost, LightGBM |
| 6 | Hyperparameter Tuning | Automated search on best model, feature importance |
| 7 | Time Series Forecasting | SARIMA, Holt-Winters, 12-month forecast |

**Dataset:** 2,641 rows x 12 columns | **Target:** `Estimated_Deliveries`

In [ ]:
import subprocess, sys
_pkgs = ['pandas', 'numpy', 'matplotlib', 'seaborn', 'scikit-learn',
         'xgboost', 'lightgbm', 'statsmodels', 'pmdarima']
for _p in _pkgs:
    try:
        __import__(_p.replace('-', '_'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', _p])
print("All packages ready.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import lightgbm as lgb
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
    'figure.dpi': 100,
    'savefig.dpi': 150,
    'figure.facecolor': 'white',
})

COLORS = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7',
          '#DDA0DD', '#98D8C8', '#F7DC6F', '#BB8FCE', '#85C1E9']
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

---
## Section 1 - Data Loading & Initial Inspection

In [ ]:
df = pd.read_csv('tesla_deliveries_dataset_2015_2025.csv')

print("="*70)
print(f"  DATASET SHAPE : {df.shape[0]:,} rows  x  {df.shape[1]} columns")
print("="*70)

print("\nColumn Info:")
print(df.dtypes.to_frame('dtype').join(
      df.isnull().sum().to_frame('nulls')).join(
      df.nunique().to_frame('unique')).to_string())

print("\nFirst 5 rows:")
df.head()

In [ ]:
df.describe().round(2)

---
## Section 2 - Data Preprocessing

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nTotal missing cells: {df.isnull().sum().sum()}")

dup_count = df.duplicated().sum()
print(f"\nDuplicate rows found: {dup_count}")
if dup_count > 0:
    df.drop_duplicates(inplace=True)
    print(f"Removed. New shape: {df.shape}")

df['Date'] = pd.to_datetime(
    df['Year'].astype(str) + '-' + df['Month'].astype(str).str.zfill(2) + '-01'
)
df.sort_values('Date', inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"\nDate column created | Range: {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"Shape after cleaning: {df.shape}")

In [ ]:
numeric_cols = ['Estimated_Deliveries', 'Production_Units', 'Avg_Price_USD',
                'Battery_Capacity_kWh', 'Range_km', 'CO2_Saved_tons', 'Charging_Stations']

print("Outlier counts (IQR x 1.5):\n" + "-"*40)
outlier_info = {}
for col in numeric_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    n_out = ((df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)).sum()
    outlier_info[col] = n_out
    print(f"  {col:30s}  ->  {n_out:4d} outliers")

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    bp = axes[i].boxplot(df[col].dropna(), patch_artist=True, widths=0.6,
                         boxprops=dict(facecolor=COLORS[i], alpha=0.7),
                         medianprops=dict(color='black', linewidth=2))
    axes[i].set_title(col.replace('_', '\n'), fontsize=10, fontweight='bold')
    axes[i].tick_params(axis='x', labelbottom=False)
axes[-1].axis('off')
fig.suptitle('Outlier Detection - Box Plots (IQR Method)', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
le_region = LabelEncoder()
le_model  = LabelEncoder()
le_source = LabelEncoder()

df['Region_Encoded'] = le_region.fit_transform(df['Region'])
df['Model_Encoded']  = le_model.fit_transform(df['Model'])
df['Source_Encoded']  = le_source.fit_transform(df['Source_Type'])

print("Label Encoding Maps:")
for name, le, col in [('Region', le_region, 'Region'),
                       ('Model', le_model, 'Model'),
                       ('Source', le_source, 'Source_Type')]:
    mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    print(f"  {name:8s} -> {mapping}")

print(f"\nPreprocessing complete | Shape: {df.shape}")

---
## Section 3 - Exploratory Data Analysis

In [ ]:
plot_cols = ['Estimated_Deliveries', 'Production_Units', 'Avg_Price_USD',
             'Battery_Capacity_kWh', 'Range_km', 'CO2_Saved_tons',
             'Charging_Stations', 'Year', 'Month']

fig, axes = plt.subplots(3, 3, figsize=(18, 13))
for i, col in enumerate(plot_cols):
    ax = axes[i // 3, i % 3]
    ax.hist(df[col], bins=35, color=COLORS[i], edgecolor='white', alpha=0.85)
    mean_val = df[col].mean()
    ax.axvline(mean_val, color='#2C3E50', linestyle='--', linewidth=1.5,
               label=f'Mean: {mean_val:,.1f}')
    ax.set_title(col, fontweight='bold')
    ax.legend(fontsize=8, loc='upper right')
    ax.set_ylabel('Frequency')
fig.suptitle('Feature Distributions', fontsize=18, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
corr_cols = ['Estimated_Deliveries', 'Production_Units', 'Avg_Price_USD',
             'Battery_Capacity_kWh', 'Range_km', 'CO2_Saved_tons', 'Charging_Stations']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(220, 20, as_cmap=True)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap=cmap, center=0,
            square=True, linewidths=1.5, cbar_kws={'shrink': 0.8},
            ax=ax, vmin=-1, vmax=1)
ax.set_title('Feature Correlation Heatmap', fontsize=16, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

print("Key correlations with Estimated_Deliveries:")
target_corr = corr['Estimated_Deliveries'].drop('Estimated_Deliveries').sort_values(ascending=False)
for feat, val in target_corr.items():
    bar = '|' * int(abs(val) * 20)
    print(f"  {feat:25s}  {val:+.3f}  {bar}")

In [ ]:
monthly_total = df.groupby('Date')['Estimated_Deliveries'].sum().reset_index()

fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(monthly_total['Date'], monthly_total['Estimated_Deliveries'],
        color='#45B7D1', linewidth=2, alpha=0.9)
ax.fill_between(monthly_total['Date'], monthly_total['Estimated_Deliveries'],
                alpha=0.15, color='#45B7D1')
ax.set_title('Total Monthly Tesla Deliveries (2015-2025)', fontsize=16, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Total Deliveries')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))
model_colors = {'Model S': '#FF6B6B', 'Model 3': '#4ECDC4', 'Model X': '#45B7D1',
                'Model Y': '#96CEB4', 'Cybertruck': '#FFEAA7'}
for model_name in df['Model'].unique():
    mdata = df[df['Model'] == model_name].groupby('Date')['Estimated_Deliveries'].sum()
    ax.plot(mdata.index, mdata.values, linewidth=2, alpha=0.85,
            label=model_name, color=model_colors.get(model_name, '#888'))
ax.set_title('Monthly Deliveries by Model', fontsize=16, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Deliveries')
ax.legend(fontsize=11, loc='upper left', framealpha=0.9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

region_del = df.groupby('Region')['Estimated_Deliveries'].sum().sort_values()
bars1 = axes[0].barh(region_del.index, region_del.values, color=COLORS[:4], edgecolor='white')
axes[0].set_title('Total Deliveries by Region', fontweight='bold', fontsize=14)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
for bar, val in zip(bars1, region_del.values):
    axes[0].text(val + region_del.max()*0.01, bar.get_y() + bar.get_height()/2,
                 f'{val/1e6:.2f}M', va='center', fontsize=10, fontweight='bold')

region_price = df.groupby('Region')['Avg_Price_USD'].mean().sort_values()
bars2 = axes[1].barh(region_price.index, region_price.values, color=COLORS[4:8], edgecolor='white')
axes[1].set_title('Average Price by Region (USD)', fontweight='bold', fontsize=14)
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
for bar, val in zip(bars2, region_price.values):
    axes[1].text(val + region_price.max()*0.01, bar.get_y() + bar.get_height()/2,
                 f'${val:,.0f}', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(data=df, x='Model', y='Estimated_Deliveries', ax=axes[0],
            palette=model_colors, linewidth=1.5, fliersize=3)
axes[0].set_title('Delivery Distribution by Model', fontweight='bold', fontsize=14)
axes[0].tick_params(axis='x', rotation=20)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))

sns.boxplot(data=df, x='Model', y='Avg_Price_USD', ax=axes[1],
            palette='Set2', linewidth=1.5, fliersize=3)
axes[1].set_title('Price Distribution by Model', fontweight='bold', fontsize=14)
axes[1].tick_params(axis='x', rotation=20)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

plt.tight_layout()
plt.show()

In [ ]:
month_names = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']
seasonal_avg = df.groupby('Month')['Estimated_Deliveries'].mean()

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(seasonal_avg.index, seasonal_avg.values, color=COLORS[:12],
              edgecolor='white', width=0.7)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_names, fontsize=11)
ax.set_title('Average Deliveries by Month - Seasonality Pattern', fontsize=16, fontweight='bold')
ax.set_ylabel('Average Deliveries')

max_month = seasonal_avg.idxmax()
bars[max_month - 1].set_edgecolor('#2C3E50')
bars[max_month - 1].set_linewidth(3)
ax.annotate(f'Peak: {month_names[max_month-1]}',
            xy=(max_month, seasonal_avg.max()), xytext=(max_month+1, seasonal_avg.max()*1.05),
            arrowprops=dict(arrowstyle='->', color='#2C3E50'),
            fontsize=12, fontweight='bold', color='#2C3E50')
plt.tight_layout()
plt.show()

In [ ]:
yearly_region = df.groupby(['Year', 'Region'])['Estimated_Deliveries'].sum().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(14, 6))
yearly_region.plot.area(ax=ax, alpha=0.75,
                        color=['#FF6B6B','#4ECDC4','#45B7D1','#96CEB4'],
                        linewidth=1.5)
ax.set_title('Yearly Deliveries by Region (Stacked Area)', fontsize=16, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Total Deliveries')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
ax.legend(title='Region', fontsize=10, title_fontsize=11, loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
pair_cols = ['Estimated_Deliveries', 'Production_Units', 'Avg_Price_USD', 'Range_km']
sample_df = df[pair_cols + ['Model']].sample(min(800, len(df)), random_state=42)
g = sns.pairplot(sample_df, hue='Model', palette=model_colors,
                 plot_kws={'alpha': 0.5, 's': 20}, diag_kind='kde',
                 height=2.5, aspect=1.1)
g.figure.suptitle('Pair Plot - Key Features by Model', fontsize=16, fontweight='bold', y=1.02)
plt.show()

---
## Section 4 - Feature Engineering

In [ ]:
df = df.sort_values(['Model', 'Region', 'Date']).reset_index(drop=True)

df['Quarter']        = (df['Month'] - 1) // 3 + 1
df['Year_Month_Idx'] = (df['Year'] - df['Year'].min()) * 12 + df['Month']
df['Sin_Month']      = np.sin(2 * np.pi * df['Month'] / 12)
df['Cos_Month']      = np.cos(2 * np.pi * df['Month'] / 12)

for lag in [1, 3, 6]:
    df[f'Delivery_Lag_{lag}'] = (
        df.groupby(['Model', 'Region'])['Estimated_Deliveries'].shift(lag)
    )

for window in [3, 6]:
    df[f'Delivery_RollMean_{window}'] = (
        df.groupby(['Model', 'Region'])['Estimated_Deliveries']
          .transform(lambda x: x.rolling(window, min_periods=1).mean())
    )
    df[f'Delivery_RollStd_{window}'] = (
        df.groupby(['Model', 'Region'])['Estimated_Deliveries']
          .transform(lambda x: x.rolling(window, min_periods=1).std())
    )

df['Price_per_kWh']             = df['Avg_Price_USD'] / df['Battery_Capacity_kWh']
df['Delivery_Production_Ratio'] = df['Estimated_Deliveries'] / df['Production_Units'].clip(lower=1)
df['Price_Range_Interaction']   = df['Avg_Price_USD'] * df['Range_km'] / 1e6

df['Delivery_Growth'] = (
    df.groupby(['Model', 'Region'])['Estimated_Deliveries'].pct_change()
)

df.fillna(0, inplace=True)

new_features = ['Quarter', 'Year_Month_Idx', 'Sin_Month', 'Cos_Month',
                'Delivery_Lag_1', 'Delivery_Lag_3', 'Delivery_Lag_6',
                'Delivery_RollMean_3', 'Delivery_RollMean_6',
                'Delivery_RollStd_3', 'Delivery_RollStd_6',
                'Price_per_kWh', 'Delivery_Production_Ratio',
                'Price_Range_Interaction', 'Delivery_Growth']

print(f"{len(new_features)} new features created")
print(f"Feature-engineered shape: {df.shape}")
print(f"\nNew features: {new_features}")
df[new_features].describe().round(3)

---
## Section 5 - Regression Modeling

Models: Linear Regression, Ridge, Lasso, Random Forest, XGBoost, LightGBM

In [ ]:
feature_cols = [
    'Production_Units', 'Avg_Price_USD', 'Battery_Capacity_kWh', 'Range_km',
    'CO2_Saved_tons', 'Charging_Stations',
    'Region_Encoded', 'Model_Encoded', 'Source_Encoded',
    'Quarter', 'Year_Month_Idx', 'Sin_Month', 'Cos_Month',
    'Delivery_Lag_1', 'Delivery_Lag_3', 'Delivery_Lag_6',
    'Delivery_RollMean_3', 'Delivery_RollMean_6',
    'Delivery_RollStd_3', 'Delivery_RollStd_6',
    'Price_per_kWh', 'Delivery_Production_Ratio',
    'Price_Range_Interaction', 'Delivery_Growth'
]
TARGET = 'Estimated_Deliveries'

X = df[feature_cols].copy()
y = df[TARGET].copy()

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)

split_idx = int(len(df) * 0.8)
X_train, X_test = X_scaled.iloc[:split_idx], X_scaled.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Train set : {X_train.shape[0]:,} rows")
print(f"Test  set : {X_test.shape[0]:,} rows")
print(f"Features  : {X_train.shape[1]}")

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression':  Ridge(alpha=1.0),
    'Lasso Regression':  Lasso(alpha=1.0, max_iter=10000),
    'Random Forest':     RandomForestRegressor(n_estimators=200, max_depth=15,
                                               random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost':           xgb.XGBRegressor(n_estimators=200, learning_rate=0.1,
                                           max_depth=6, random_state=RANDOM_STATE,
                                           verbosity=0, n_jobs=-1),
    'LightGBM':          lgb.LGBMRegressor(n_estimators=200, learning_rate=0.1,
                                            max_depth=6, random_state=RANDOM_STATE,
                                            verbose=-1, n_jobs=-1),
}

results = {}
predictions = {}

print("="*85)
print(f"{'Model':25s} | {'R2':>8s} | {'MAE':>10s} | {'RMSE':>10s} | {'MAPE %':>8s}")
print("="*85)

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    predictions[name] = y_pred

    r2   = r2_score(y_test, y_pred)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mape = np.mean(np.abs((y_test - y_pred) / y_test.clip(lower=1))) * 100

    results[name] = {'R2': r2, 'MAE': mae, 'RMSE': rmse, 'MAPE (%)': mape}
    print(f"{name:25s} | {r2:8.4f} | {mae:10.1f} | {rmse:10.1f} | {mape:7.1f}%")

print("="*85)

results_df = pd.DataFrame(results).T.sort_values('R2', ascending=False)
print("\nBest model:", results_df.index[0],
      f"(R2 = {results_df.iloc[0]['R2']:.4f})")

In [ ]:
top3 = results_df.head(3).index.tolist()

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for i, name in enumerate(top3):
    ax = axes[i]
    ax.scatter(y_test, predictions[name], alpha=0.45, s=18, color=COLORS[i],
               edgecolors='white', linewidth=0.3)
    lo, hi = y_test.min(), y_test.max()
    ax.plot([lo, hi], [lo, hi], '--', color='#2C3E50', linewidth=2, label='Perfect Fit')
    ax.set_title(f'{name}\nR2 = {results[name]["R2"]:.4f}  |  RMSE = {results[name]["RMSE"]:.0f}',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Actual Deliveries')
    ax.set_ylabel('Predicted Deliveries')
    ax.legend(fontsize=9)
fig.suptitle('Actual vs Predicted - Top 3 Models', fontsize=17, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

r2_vals = results_df['R2'].sort_values()
axes[0].barh(r2_vals.index, r2_vals.values, color=COLORS[:len(r2_vals)], edgecolor='white')
axes[0].set_title('R2 Score Comparison', fontweight='bold', fontsize=14)
axes[0].set_xlabel('R2 Score')
for i, (idx, val) in enumerate(r2_vals.items()):
    axes[0].text(val + 0.005, i, f'{val:.4f}', va='center', fontweight='bold', fontsize=10)

rmse_vals = results_df['RMSE'].sort_values(ascending=False)
axes[1].barh(rmse_vals.index, rmse_vals.values, color=COLORS[:len(rmse_vals)], edgecolor='white')
axes[1].set_title('RMSE Comparison (lower = better)', fontweight='bold', fontsize=14)
axes[1].set_xlabel('RMSE')
for i, (idx, val) in enumerate(rmse_vals.items()):
    axes[1].text(val + rmse_vals.max()*0.01, i, f'{val:.0f}', va='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
best_name = results_df.index[0]
y_pred_best = predictions[best_name]
residuals = y_test.values - y_pred_best

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

axes[0].scatter(y_pred_best, residuals, alpha=0.4, s=18, color='#45B7D1',
                edgecolors='white', linewidth=0.3)
axes[0].axhline(y=0, color='#FF6B6B', linestyle='--', linewidth=2)
axes[0].set_title(f'Residual Plot - {best_name}', fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Residual')

axes[1].hist(residuals, bins=40, color='#96CEB4', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='#FF6B6B', linestyle='--', linewidth=2)
axes[1].set_title('Residual Distribution', fontweight='bold')
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Count')

sorted_res = np.sort(residuals)
theoretical = np.sort(np.random.normal(np.mean(residuals), np.std(residuals), len(residuals)))
axes[2].scatter(theoretical, sorted_res, alpha=0.4, s=12, color='#DDA0DD')
lims = [min(theoretical.min(), sorted_res.min()), max(theoretical.max(), sorted_res.max())]
axes[2].plot(lims, lims, '--', color='#2C3E50', linewidth=2)
axes[2].set_title('Q-Q Plot (Residuals vs Normal)', fontweight='bold')
axes[2].set_xlabel('Theoretical Quantiles')
axes[2].set_ylabel('Observed Quantiles')

plt.tight_layout()
plt.show()

print(f"Residual stats for {best_name}:")
print(f"   Mean : {np.mean(residuals):,.1f}")
print(f"   Std  : {np.std(residuals):,.1f}")
print(f"   Skew : {pd.Series(residuals).skew():.3f}")

---
## Section 6 - Hyperparameter Tuning

In [ ]:
best_name = results_df.index[0]
print(f"Tuning: {best_name}\n")

if 'XGBoost' in best_name:
    param_grid = {
        'n_estimators':      [100, 200, 300, 500],
        'max_depth':         [3, 5, 7, 9],
        'learning_rate':     [0.01, 0.05, 0.1, 0.2],
        'subsample':         [0.7, 0.8, 0.9, 1.0],
        'colsample_bytree':  [0.7, 0.8, 0.9, 1.0],
        'min_child_weight':  [1, 3, 5],
    }
    base_model = xgb.XGBRegressor(random_state=RANDOM_STATE, verbosity=0, n_jobs=-1)

elif 'LightGBM' in best_name:
    param_grid = {
        'n_estimators':      [100, 200, 300, 500],
        'max_depth':         [3, 5, 7, -1],
        'learning_rate':     [0.01, 0.05, 0.1, 0.2],
        'subsample':         [0.7, 0.8, 0.9, 1.0],
        'colsample_bytree':  [0.7, 0.8, 0.9, 1.0],
        'num_leaves':        [15, 31, 63, 127],
    }
    base_model = lgb.LGBMRegressor(random_state=RANDOM_STATE, verbose=-1, n_jobs=-1)

elif 'Random Forest' in best_name:
    param_grid = {
        'n_estimators':      [100, 200, 300, 500],
        'max_depth':         [5, 10, 15, 20, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf':  [1, 2, 4],
        'max_features':      ['sqrt', 'log2', None],
    }
    base_model = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)

else:
    param_grid = {'alpha': [0.01, 0.1, 1, 10, 100, 1000]}
    base_model = Ridge()

print(f"Parameter grid ({len(param_grid)} params):")
for k, v in param_grid.items():
    print(f"  {k}: {v}")

In [ ]:
search = RandomizedSearchCV(
    base_model,
    param_distributions=param_grid,
    n_iter=30,
    cv=5,
    scoring='r2',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=0,
    return_train_score=True,
)
search.fit(X_train, y_train)

print(f"Best CV R2: {search.best_score_:.4f}\n")
print("Best Parameters:")
for k, v in search.best_params_.items():
    print(f"  {k:25s} = {v}")

y_pred_tuned = search.best_estimator_.predict(X_test)
r2_tuned   = r2_score(y_test, y_pred_tuned)
mae_tuned  = mean_absolute_error(y_test, y_pred_tuned)
rmse_tuned = np.sqrt(mean_squared_error(y_test, y_pred_tuned))
mape_tuned = np.mean(np.abs((y_test - y_pred_tuned) / y_test.clip(lower=1))) * 100

print("\n" + "="*60)
print(f"{'Metric':<12} {'Before Tuning':>15} {'After Tuning':>15} {'Delta':>10}")
print("="*60)
for metric, before, after in [
    ('R2',       results[best_name]['R2'],      r2_tuned),
    ('MAE',      results[best_name]['MAE'],      mae_tuned),
    ('RMSE',     results[best_name]['RMSE'],     rmse_tuned),
    ('MAPE (%)', results[best_name]['MAPE (%)'], mape_tuned),
]:
    delta = after - before
    sign = '+' if delta > 0 else ''
    print(f"{metric:<12} {before:>15.4f} {after:>15.4f} {sign}{delta:>9.4f}")
print("="*60)

In [ ]:
if hasattr(search.best_estimator_, 'feature_importances_'):
    importances = pd.Series(
        search.best_estimator_.feature_importances_, index=feature_cols
    ).sort_values(ascending=True)
    top15 = importances.tail(15)

    fig, ax = plt.subplots(figsize=(10, 8))
    bars = ax.barh(top15.index, top15.values, color=COLORS[:15][::-1], edgecolor='white')
    ax.set_title(f'Top 15 Feature Importances - {best_name} (Tuned)',
                 fontsize=14, fontweight='bold')
    ax.set_xlabel('Importance Score')

    for bar, val in zip(bars, top15.values):
        ax.text(val + top15.max()*0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=9)

    plt.tight_layout()
    plt.show()
else:
    coefs = pd.Series(search.best_estimator_.coef_, index=feature_cols).abs().sort_values()
    top15 = coefs.tail(15)
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(top15.index, top15.values, color=COLORS[:15][::-1], edgecolor='white')
    ax.set_title(f'Top 15 Feature Coefficients (abs) - {best_name}',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

---
## Section 7 - Time Series Forecasting

Methods: SARIMA, Holt-Winters Exponential Smoothing

In [ ]:
ts_data = df.groupby('Date')['Estimated_Deliveries'].sum().reset_index()
ts_data.set_index('Date', inplace=True)
ts_data = ts_data.asfreq('MS')
ts_data = ts_data.interpolate(method='linear')

print(f"Time series range: {ts_data.index[0].date()} to {ts_data.index[-1].date()}")
print(f"Total months: {len(ts_data)}")

fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(ts_data.index, ts_data['Estimated_Deliveries'], color='#45B7D1', linewidth=2)
ax.fill_between(ts_data.index, ts_data['Estimated_Deliveries'], alpha=0.15, color='#45B7D1')
ax.set_title('Monthly Total Tesla Deliveries - Time Series', fontsize=16, fontweight='bold')
ax.set_ylabel('Total Deliveries')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
plt.tight_layout()
plt.show()

In [ ]:
adf_result = adfuller(ts_data['Estimated_Deliveries'])

print("Augmented Dickey-Fuller Test")
print("="*45)
print(f"  ADF Statistic  : {adf_result[0]:.4f}")
print(f"  P-value        : {adf_result[1]:.6f}")
print(f"  Lags Used      : {adf_result[2]}")
print(f"  Observations   : {adf_result[3]}")
print(f"  Critical Values:")
for key, val in adf_result[4].items():
    print(f"    {key:>5s}: {val:.4f}")
print(f"\n  Result: {'Stationary' if adf_result[1] < 0.05 else 'Non-stationary (may need differencing)'}")

In [ ]:
decomp = seasonal_decompose(ts_data['Estimated_Deliveries'], model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(15, 12), sharex=True)
components = [
    ('Observed',  decomp.observed, '#45B7D1'),
    ('Trend',     decomp.trend,    '#FF6B6B'),
    ('Seasonal',  decomp.seasonal, '#4ECDC4'),
    ('Residual',  decomp.resid,    '#96CEB4'),
]
for ax, (title, data, color) in zip(axes, components):
    ax.plot(data.index, data.values, color=color, linewidth=1.5)
    if title == 'Residual':
        ax.fill_between(data.index, data.values, alpha=0.3, color=color)
    ax.set_title(title, fontsize=13, fontweight='bold', loc='left')
    ax.grid(True, alpha=0.3)
fig.suptitle('Time Series Decomposition (Additive)', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
n_forecast = 12
train_ts = ts_data.iloc[:-n_forecast]
test_ts  = ts_data.iloc[-n_forecast:]

print(f"Training: {train_ts.index[0].date()} to {train_ts.index[-1].date()}  ({len(train_ts)} months)")
print(f"Testing : {test_ts.index[0].date()} to {test_ts.index[-1].date()}  ({len(test_ts)} months)")

In [ ]:
try:
    import pmdarima as pm
    print("Running auto_arima...")
    auto_model = pm.auto_arima(
        train_ts['Estimated_Deliveries'],
        seasonal=True, m=12,
        suppress_warnings=True,
        stepwise=True,
        error_action='ignore',
        max_p=3, max_q=3, max_P=2, max_Q=2,
        max_d=2, max_D=1,
    )
    order = auto_model.order
    seasonal_order = auto_model.seasonal_order
    print(f"auto_arima selected: SARIMA{order}x{seasonal_order}")
except ImportError:
    print("pmdarima not installed. Using default SARIMA(1,1,1)(1,1,1,12)")
    order = (1, 1, 1)
    seasonal_order = (1, 1, 1, 12)

sarima_model = SARIMAX(
    train_ts['Estimated_Deliveries'],
    order=order,
    seasonal_order=seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False,
)
sarima_fit = sarima_model.fit(disp=False)

sarima_forecast = sarima_fit.get_forecast(steps=n_forecast)
sarima_mean = sarima_forecast.predicted_mean
sarima_ci   = sarima_forecast.conf_int()

sarima_rmse = np.sqrt(mean_squared_error(test_ts['Estimated_Deliveries'], sarima_mean))
sarima_mae  = mean_absolute_error(test_ts['Estimated_Deliveries'], sarima_mean)

print(f"\nSARIMA Forecast Evaluation:")
print(f"   RMSE: {sarima_rmse:,.1f}")
print(f"   MAE : {sarima_mae:,.1f}")

In [ ]:
hw_model = ExponentialSmoothing(
    train_ts['Estimated_Deliveries'],
    seasonal_periods=12,
    trend='add',
    seasonal='add',
    use_boxcox=False,
).fit(optimized=True)

hw_forecast = hw_model.forecast(n_forecast)

hw_rmse = np.sqrt(mean_squared_error(test_ts['Estimated_Deliveries'], hw_forecast))
hw_mae  = mean_absolute_error(test_ts['Estimated_Deliveries'], hw_forecast)

print(f"Holt-Winters Forecast Evaluation:")
print(f"   RMSE: {hw_rmse:,.1f}")
print(f"   MAE : {hw_mae:,.1f}")

print(f"\n{'='*50}")
print(f"{'Model':25s} {'RMSE':>12s} {'MAE':>12s}")
print(f"{'='*50}")
print(f"{'SARIMA':25s} {sarima_rmse:>12,.1f} {sarima_mae:>12,.1f}")
print(f"{'Holt-Winters':25s} {hw_rmse:>12,.1f} {hw_mae:>12,.1f}")
winner = 'SARIMA' if sarima_rmse < hw_rmse else 'Holt-Winters'
print(f"{'='*50}")
print(f"Winner: {winner}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

axes[0].plot(train_ts.index, train_ts['Estimated_Deliveries'],
             color='#45B7D1', linewidth=1.5, label='Training Data')
axes[0].plot(test_ts.index, test_ts['Estimated_Deliveries'],
             color='#2C3E50', linewidth=2.5, label='Actual', marker='o', markersize=5)
axes[0].plot(sarima_mean.index, sarima_mean.values,
             color='#FF6B6B', linewidth=2, linestyle='--', label='SARIMA Forecast',
             marker='s', markersize=4)
axes[0].fill_between(sarima_ci.index,
                     sarima_ci.iloc[:, 0], sarima_ci.iloc[:, 1],
                     alpha=0.15, color='#FF6B6B', label='95% CI')
axes[0].set_title(f'SARIMA Forecast  (RMSE={sarima_rmse:,.0f})',
                  fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))

axes[1].plot(train_ts.index, train_ts['Estimated_Deliveries'],
             color='#45B7D1', linewidth=1.5, label='Training Data')
axes[1].plot(test_ts.index, test_ts['Estimated_Deliveries'],
             color='#2C3E50', linewidth=2.5, label='Actual', marker='o', markersize=5)
axes[1].plot(hw_forecast.index, hw_forecast.values,
             color='#4ECDC4', linewidth=2, linestyle='--', label='Holt-Winters Forecast',
             marker='D', markersize=4)
axes[1].set_title(f'Holt-Winters Forecast  (RMSE={hw_rmse:,.0f})',
                  fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].set_xlabel('Date')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))

fig.suptitle('12-Month Forecast Comparison', fontsize=17, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
best_ts_model_name = 'SARIMA' if sarima_rmse < hw_rmse else 'Holt-Winters'

if best_ts_model_name == 'SARIMA':
    full_sarima = SARIMAX(
        ts_data['Estimated_Deliveries'],
        order=order,
        seasonal_order=seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False,
    ).fit(disp=False)
    future = full_sarima.get_forecast(steps=12)
    future_mean = future.predicted_mean
    future_ci   = future.conf_int()
else:
    full_hw = ExponentialSmoothing(
        ts_data['Estimated_Deliveries'],
        seasonal_periods=12, trend='add', seasonal='add',
    ).fit(optimized=True)
    future_mean = full_hw.forecast(12)
    residuals_hw = ts_data['Estimated_Deliveries'] - full_hw.fittedvalues
    std_resid = residuals_hw.std()
    future_ci = pd.DataFrame({
        'lower': future_mean - 1.96 * std_resid,
        'upper': future_mean + 1.96 * std_resid,
    }, index=future_mean.index)

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(ts_data.index, ts_data['Estimated_Deliveries'],
        color='#45B7D1', linewidth=2, label='Historical Data')
ax.plot(future_mean.index, future_mean.values,
        color='#FF6B6B', linewidth=2.5, linestyle='--', label=f'{best_ts_model_name} - Future Forecast',
        marker='o', markersize=5)

if best_ts_model_name == 'SARIMA':
    ax.fill_between(future_ci.index,
                    future_ci.iloc[:, 0], future_ci.iloc[:, 1],
                    alpha=0.15, color='#FF6B6B', label='95% Confidence Interval')
else:
    ax.fill_between(future_ci.index,
                    future_ci['lower'], future_ci['upper'],
                    alpha=0.15, color='#FF6B6B', label='95% Confidence Interval')

ax.axvline(ts_data.index[-1], color='#2C3E50', linestyle=':', linewidth=1.5, alpha=0.5)
ax.set_title(f'12-Month Future Forecast - {best_ts_model_name}', fontsize=16, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Total Deliveries')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f"\nForecasted Monthly Deliveries ({best_ts_model_name}):")
for date, val in future_mean.items():
    print(f"  {date.strftime('%B %Y'):>18s}  ->  {val:>12,.0f}")

---
## Conclusion

| Stage | Summary |
|-------|---------|
| Preprocessing | Date column, 3 label encodings, IQR outlier analysis |
| EDA | 10+ visualisations - distributions, trends, regions, seasonality |
| Feature Engineering | 15 new features - lags, rolling stats, cyclical, interactions |
| Regression | 6 models evaluated, best selected by R2 |
| Hyperparameter Tuning | RandomizedSearchCV (30 iterations, 5-fold CV) |
| Time Series | SARIMA + Holt-Winters, 12-month forecast with confidence intervals |